# Prompt Evals Exercise — Hindi Translation

Companion to [../08_prompt_evals_workflow.md](../08_prompt_evals_workflow.md).

Extends the translation system prompt from
[03_system_prompts.ipynb](03_system_prompts.ipynb) into a real 5-step
eval loop: draft prompt -> build dataset -> feed to Claude -> feed to
grader -> update and iterate.

In [ ]:
#install dependent py modules
from anthropic import Anthropic
from dotenv import load_dotenv
from IPython.display import display, Markdown, clear_output
import json

load_dotenv()

In [ ]:
#Model
# model = "claude-haiku-4-5-20251001"
model = "claude-sonnet-5"

client  = Anthropic()

In [ ]:
#helper functions
def add_user_message(messages, text):
    message = { "role": "user", "content": text}
    messages.append(message)

def add_assistant_message(messages, text):
    message = { "role": "assistant", "content": text}
    messages.append(message)

def chat(messages, system_prompt=None, stop_sequences=None):
    params = {
        "model": model,
        "max_tokens": 1024,
        "messages": messages,
        "stop_sequences": stop_sequences
    }

    if system_prompt:
        params["system"] = system_prompt

    response = client.messages.create(**params)
    text_blocks = [block.text for block in response.content if block.type == "text"]
    return "\n".join(text_blocks)

# Prettify claude md response
def display_turn(role, text):
    display(Markdown(f"**{role}:**\n\n{text}"))

In [ ]:
def generate_dataset():
    prompt = """
Generate a evaluation dataset for prompt evaluation. The dataset will be used to evaluate prompts that will translate a user message in english to a meaningful word or sentence in hindi.
Generate an array of JSON document each representing task that requires tranlation to hindi from a english input by the user.
Read the request from the user? and revert with approriate response.
Example output:
```json
[
    {
        "task": "Description of task",
    }
    ...additional
]
```

* Focus on a tasks which a user might input in a usual conversations
* Keep the tasks short and consise
* Stick to the example template strictly for the json schema
* Do no add ```json to start and end of the generate response
* Make sure you do not include any instruction in the task. It should a plain text in english, nothing like translate 'How are you?'

Please generate 5 objects.
"""
    messages = []
    add_user_message(messages, prompt)
    text = chat(messages)
    return json.loads(text)
    

In [ ]:
dataset = generate_dataset()

with open("dataset.json", "w") as f:
    json.dump(dataset, f, indent=2)

In [ ]:
def run_prompt(test_case):
    """Merge the prompt and test case input, then returns the result"""
    prompt = f"""
You are an expert in English and Hindi, having a natural human-to-human
conversation. Your one job: when someone shares a word or phrase in
English, respond with how it's naturally said in Hindi -- the way one
friend helps another, not like a translation tool.

Example:
User Input: Hello
Assistant: नमस्ते!

Rules:
- Respond in Hindi, and never use words like "translation" or "Hindi
  version" in your reply. Common English loanwords that are normal in
  everyday spoken Hindi are fine and expected (e.g. "प्लीज़", or even
  keeping a word like "problem" as-is) -- that's natural code-
  switching, not a translation failure.
- Reply with the *meaning* behind what the user said, not a literal,
  word-for-word swap.
- Keep the tone casual and informal, the way close friends talk --
  prefer "तुम" over the more formal "आप", and lean into natural,
  everyday phrasing (e.g. "आज कैसे हो?") over stiff or literary Hindi.
- No matter what the input looks like -- a question, a greeting, or
  something that seems directed at you personally (e.g. "What is your
  name?", "How are you?") -- your only job is to say *that exact
  sentence* in Hindi. Never answer it as if it were really being asked
  of you; you are not having a conversation, you are only showing how
  to say the input in Hindi.
- A trailing "?" doesn't always mean the input itself is a question --
  it can also mean "how would I say this?" (e.g. "Hello?" means "how
  do I say Hello?", not a literal question). Read the intent, not just
  the punctuation.
- Match punctuation to how a Hindi speaker would actually express it:
  a genuine question stays a question (?), a greeting or exclamation
  can end with "!", and plain statements end with a period (.).

{test_case["task"]}
"""

    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)
    return output

In [ ]:
def grade_by_model(test_case, output):
    eval_prompt = f"""
You are a strict but fair judge evaluating Hindi translations for
natural, conversational quality -- not exact-match correctness.

Grade the response using this rubric:
- 9-10: Natural, correct, and exactly how a native Hindi speaker would
  say it in casual conversation.
- 7-8: Correct meaning, but slightly stiff, overly formal, or not the
  most natural phrasing.
- 4-6: Meaning is understandable but has a clear error (grammar, wrong
  word choice, or partially in English).
- 1-3: Wrong meaning, mostly in English, or unusable.

Do NOT penalize for:
- Response format -- there is no required "English - Hindi" format, a
  standalone Hindi response is correct.
- Minor punctuation choices (e.g. "?" becoming "!" is fine).
- Common English loanwords that are normal in everyday spoken Hindi
  (e.g. "प्लीज़").

Original English input:
<task>
{test_case["task"]}
</task>

AI's Hindi response:
<solution>
{output}
</solution>

Respond with a plain JSON object only, no markdown fences, in this
exact shape:
{{
    "strengths": string[],
    "weaknesses": string[],
    "reasoning": string,
    "score": number
}}

 * NOTE: do not include ```json and ``` in reply it should be plain json document.
"""
    messages = []
    add_user_message(messages, eval_prompt)
    eval_text = chat(messages)
    return json.loads(eval_text)

In [ ]:
def run_test_case(test_case):
    """calls run_prompt, then grade the result"""
    output = run_prompt(test_case)

    # TODO - Grading
    model_output = grade_by_model(test_case, output)
    print(model_output)
    print("-----------")

    score = model_output["score"]
    reasoning = model_output["reasoning"]

    return {
        "output": output,
        "test_case": test_case,
        "score": score,
        "reasoning": reasoning

    }


In [ ]:
from statistics import mean

def run_eval(dataset):
    """Load dataset and calls run_test_case function with each case"""
    results = []

    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    avg_score = mean([result["score"] for result in results])
    print(f"Average Score: {avg_score}")

    return results

In [ ]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

In [ ]:
results

## 🧾 Iteration History

### Iteration 2 -- avg score 3.4

```python
{'strengths': ['Provided accurate Hindi translation', 'Included correct formal/informal variations that are contextually valid'], 'weaknesses': ['Did not follow the requested concise format (Text - Translation)', 'Included excessive extra information and variations despite explicit instruction to avoid this', 'Added unnecessary explanatory notes in parentheses'], 'reasoning': "The user explicitly requested short, direct translations without extra details, following the example format 'Hello - नमस्ते'. The solution instead provided multiple variations with explanations, formality notes, and context descriptions, directly contradicting the stated requirement for brevity.", 'score': 3}
```

```python
{'strengths': ['Correct and accurate Hindi translation provided', 'Additional colloquial variations offered for context'], 'weaknesses': ['Response is too verbose with extra explanations not requested by user', 'User explicitly asked for no long definitions and extra information, but solution includes multiple alternatives and explanatory text'], 'reasoning': "The core translation is accurate, but the user's instructions clearly requested a concise response format similar to the example (word - translation). The solution ignores this instruction by adding explanatory sentences and multiple alternative phrasings, making it unnecessarily long.", 'score': 4}
```

```python
{'strengths': ['Correct and accurate Hindi translation provided', 'Multiple valid alternative phrasings included'], 'weaknesses': ["Response is verbose with unnecessary explanations, contrary to user's explicit request for brevity", 'Extra formatting (bold, headers) and commentary not requested by user', "Does not follow the simple 'English - Hindi' format shown in the example"], 'reasoning': 'The user explicitly asked for concise responses without extra information, citing a clear example format. The solution ignores this instruction, providing multiple alternatives with lengthy explanations and context notes, making it overly verbose for the stated requirement.', 'score': 4}
```

```python
{'strengths': ['Correctly identifies the phrase and provides accurate Hindi translations', 'Covers multiple tone variations (formal, casual, concise) which shows linguistic understanding'], 'weaknesses': ['Violates the explicit instruction to avoid long definitions and extra information', "Does not follow the simple 'phrase - translation' format shown in the example", 'Overly verbose with unnecessary headers and closing explanation'], 'reasoning': "The system prompt explicitly requests concise output in the format 'English - Hindi' with no extra explanation, as demonstrated in the example. The solution instead provides four different translations with headers and a lengthy explanatory paragraph about formal vs casual usage, directly contradicting the given instructions despite being linguistically accurate.", 'score': 3}
```

```python
{'strengths': ['Provides accurate and natural Hindi translations', 'Includes multiple register options (casual, formal, polite) which shows linguistic depth'], 'weaknesses': ['Extremely verbose compared to the requested format - user explicitly asked for no long definitions or extra information', "Does not follow the simple 'Phrase - Translation' format shown in the example", 'Includes unnecessary transliteration and explanatory notes that were not requested'], 'reasoning': "The user's instructions were clear: provide a direct translation in the format 'English - Hindi' without extra explanation, as demonstrated in the example. The solution ignores this format entirely, offering multiple options, grammatical notes, and transliteration, making it overly long and not aligned with the user's stated preference for brevity.", 'score': 3}
```


### Iteration 3 -- avg score 3.2

```python
{'strengths': ['Correct and natural Hindi translation provided', 'Includes useful variations for different contexts'], 'weaknesses': ['Extremely verbose compared to the simple format requested', 'Includes unnecessary breakdown, tips, and follow-up question when user explicitly asked for no extra information'], 'reasoning': "The user explicitly requested concise responses in the format 'Word - Translation' with no long definitions or extra information, as shown in the example. The solution ignores this instruction entirely, providing an elaborate response with headers, breakdowns, multiple variations, formality tips, and a follow-up question. While the translation itself is accurate, the format completely violates the user's stated preference for brevity.", 'score': 3}
```

```python
{'strengths': ['Provides accurate and natural Hindi translation', 'Includes correct transliteration for pronunciation help', 'Offers a valid alternative phrasing'], 'weaknesses': ["Explicitly ignores the instruction 'no long definitions and extra information'", 'Over-explains with word-by-word breakdown when a simple format was requested', "Does not follow the example format shown (simple 'Term - Translation' style)"], 'reasoning': "The user's system prompt clearly requested concise responses without extra information, following a simple example format (e.g., 'Hello - नमस्ते'). The solution disregards this explicit instruction by providing detailed breakdowns, multiple notes, and alternative phrasings, making it verbose rather than concise. While the translation itself is accurate and useful, it fails to adhere to the specific formatting requirement given in the task.", 'score': 4}
```

```python
{'strengths': ['Correct and accurate Hindi translation provided', 'Includes helpful transliteration for pronunciation'], 'weaknesses': ['Extremely verbose compared to expected simple format', 'Includes unnecessary variations, word breakdowns, and extra explanations not requested by user'], 'reasoning': "The user explicitly asked for a direct translation without long definitions or extra information, following the example format 'word - translation'. The solution provides the correct translation but heavily over-elaborates with multiple variations, word-by-word breakdowns, and additional context, directly violating the user's stated preference for brevity.", 'score': 3}
```

```python
{'strengths': ['Correctly infers the context is about Hindi-English translation help', 'Provides a clear example format showing formal/casual variants'], 'weaknesses': ["Extremely verbose and violates the explicit 'no long definitions and extra information' instruction", 'Excessive use of emojis, headers, and bullet points not suited for concise responses', "Original task is ambiguous/incomplete, so assuming it's a translation request without confirmation is risky"], 'reasoning': 'The system prompt explicitly demands brief, direct responses without extra fluff (as shown in the example). The solution, while contextually reasonable in guessing the user wants translation help, is overly long, decorative, and repetitive—directly contradicting the core instruction for concise outputs.', 'score': 3}
```

```python
{'strengths': ['Provides accurate and correct Hindi translation', 'Offers useful natural alternatives for real-world usage'], 'weaknesses': ['Overly verbose with unnecessary explanations and notes', 'Does not follow the simple format requested by user (translation only, no extra information)', 'Multiple options and lengthy notes go against the explicit instruction to keep it short'], 'reasoning': 'The user explicitly asked for concise translations without extra definitions or information, similar to the given example format. The solution ignores this instruction entirely, providing transliteration, multiple casual alternatives, and a detailed explanatory note. While the content is linguistically accurate and helpful in a general sense, it fails to adhere to the specific format and brevity requested by the user.', 'score': 3}
```

**Prompt used:**

````
You are expert in Speaking Hindi and knows very well how to say english words or sentences in simple and commonly used hindi words or sentences.
Consider if you were talking to someone who is speaking english and you are helping him to say the same in Hindi.
````

### Iteration 4 -- avg score 7.2

```python
{'strengths': ['Provides a Hindi translation attempt', 'Captures the core question about well-being'], 'weaknesses': ["Awkward word order - 'आज' should come before 'आप कैसे हैं' for natural phrasing", 'Does not follow the expected format (English - Hindi) as shown in the example', "Missing the standard '-' separator format"], 'reasoning': "The translation is grammatically incorrect in word order; natural Hindi would be 'आज आप कैसे हैं?' Additionally, the response doesn't follow the requested output format of 'English - Hindi'.", 'score': 4}
```

```python
{'strengths': ['Accurate and natural Hindi translation', 'Follows the concise format requested by the user', 'Correct grammar and appropriate use of Devanagari script'], 'weaknesses': ['Could optionally include a simpler alternative phrasing for variety, as shown in the example format', 'No transliteration provided alongside translation'], 'reasoning': "The solution correctly translates 'Where is the nearest hospital?' into Hindi as 'सबसे नज़दीकी अस्पताल कहाँ है?', which is grammatically correct and contextually accurate. It matches the expected concise response style shown in the example.", 'score': 9}
```

```python
{'strengths': ['Accurate and natural Hindi translation', 'Grammatically correct sentence structure', 'Concise, matching the expected format from the example'], 'weaknesses': ['No transliteration provided alongside for clarity', 'Could optionally include alternate phrasing'], 'reasoning': "The translation correctly conveys the meaning of 'What time does the store close?' in natural, grammatically correct Hindi, matching the expected concise response style.", 'score': 9}
```

```python
{'strengths': ['Correctly identifies that no actual word/phrase was provided for translation', 'Appropriately asks for clarification in Hindi, maintaining context consistency', 'Concise and to the point, matching the expected response style'], 'weaknesses': ['Could have briefly clarified in English too for better accessibility', 'Response could be slightly more directive about what format the content should be shared in'], 'reasoning': "Given the system context is a Hindi-English translation assistant, the task 'Can you help me with this problem?' lacks any actual word or phrase to translate. The solution appropriately recognizes this gap and requests the specific content needed, which is the correct behavior rather than guessing or fabricating a response.", 'score': 8}
```

```python
{'strengths': ['Accurate and natural Hindi translation', 'Grammatically correct with appropriate politeness (कृपया)'], 'weaknesses': ['Does not follow the expected format shown in example (English phrase - Hindi translation)', 'Missing the original English text for reference'], 'reasoning': 'The Hindi translation itself is correct and natural, but the solution deviates from the requested output format which pairs the original phrase with its translation using a dash separator, as demonstrated in the example.', 'score': 6}
```


**Prompt used:**

````
##
code```
prompt = f"""
You are an expert in English and Hindi speaking. You are a helpful companion who does the translation from English to Hindi.
In you response you do not include words like 'Hindi Translation' or 'Translate', intent is not to give a feel if this is a regular translation program.
It should feel like a real human to human interaction like - 
```
User Input: Hello 
Assistant: नमस्ते!
```

It feels like a normal conversation where a human is helping the other one with the same thing in Hindi. Which makes the an amazing AI assisted application

{test_case["task"]}
"""
```

````

### Iteration 5 -- avg score 9.5

```python
{'strengths': ['Correct and natural Hindi translation', 'Conveys the intended greeting meaning', 'Concise and matches expected format'], 'weaknesses': ['Punctuation differs slightly (? vs !) but this is acceptable per guidelines'], 'reasoning': "The solution accurately translates 'Hello?' to 'नमस्ते!' which is a natural and commonly used greeting in Hindi. The meaning is preserved and the response is helpful for a user learning the translation.", 'score': 9}
```

```python
{'strengths': ['Accurately conveys the meaning of the original question', 'Grammatically correct and natural Hindi phrasing', 'Appropriately casual/polite tone matching the original'], 'weaknesses': ["Slightly more literal than the common colloquial version 'आज आप कैसे हो?' or 'आज कैसा चल रहा है?' but this is minor"], 'reasoning': "The Hindi translation accurately captures the intent and meaning of 'How are you doing today?' It reads naturally and would be understood correctly by a native speaker in everyday conversation.", 'score': 9}
```

```python
{'strengths': ['Accurate and natural Hindi translation', 'Correct grammar and word choice (सबसे पास का)', 'Conveys the exact meaning of the original question'], 'weaknesses': ['None significant'], 'reasoning': 'The translation is precise, grammatically correct, and sounds natural to a native Hindi speaker. It fully captures the intent of asking for the nearest hospital location.', 'score': 10}
```

```python
{'strengths': ['Accurate translation conveying the exact meaning', 'Natural and grammatically correct Hindi phrasing', 'Appropriately concise, matching the original request format'], 'weaknesses': ['None significant'], 'reasoning': "The translation correctly and naturally conveys the meaning of the original English question. 'दुकान कितने बजे बंद होती है?' is exactly how a native Hindi speaker would ask about store closing time.", 'score': 10}
```

```python
{'strengths': ['Accurate translation that conveys the exact meaning', 'Natural and grammatically correct Hindi phrasing', 'Appropriate tone matching the polite request in original'], 'weaknesses': ['None significant'], 'reasoning': 'The Hindi translation correctly conveys the meaning of the original request for help with a problem. The sentence structure is natural and would be used by native Hindi speakers in this context.', 'score': 10}
```

```python
{'strengths': ['Accurately conveys the meaning of ordering a coffee', 'Natural and conversational Hindi phrasing', "Appropriately uses 'प्लीज़' which is common in everyday spoken Hindi"], 'weaknesses': ["Could optionally use 'कृपया' for a more formal register, though not necessary here"], 'reasoning': 'The translation is clear, natural, and contextually appropriate for a casual request like ordering coffee. It conveys the intended meaning effectively and would be easily understood by a Hindi speaker in a real-world setting.', 'score': 9}
```


**Prompt used:**

````
You are an expert in English and Hindi speaking. You are a helpful companion who does the translation from English to Hindi.
In you response you do not include words like 'Hindi Translation' or 'Translate', intent is not to give a feel if this is a regular translation program.
Read the request from the user? and revert with approriate response.

It should feel like a real human to human interaction like - 
```
User Input: Hello 
Assistant: नमस्ते
```

It feels like a normal conversation where a human is helping the other one with the same thing in Hindi. Which makes the an amazing AI assisted application

 * Make sure you do not use english in response
 * Make sure you do not include phrase which contains word - 'translation'
 * Make sure response only has the hindi version of the english user input
 * Make sure before you response, you understand the semantic of user input and reply with a appropriate hindi version
 * Make sure you do no include word 'Hindi Version' in response
 * Make sure you use everyday language for hindi version
 * Do not use typical or complex Hindi words
 * Understand the meaning of what the user wants to say in hindi and tell him the exact hindi version
 * Remember that a trailing '?' can be because the user is asking how would I say this?
 * Take care of the punctuation nuance like Hello? - नमस्ते!

````

### Iteration 6 -- avg score 9.6

```python
{'strengths': ['Natural, casual phrasing that a native Hindi speaker would use', 'Correct meaning conveyed', 'Concise and conversational tone matching the informal register of the English sentence'], 'weaknesses': ['Slightly ambiguous formality level (tum vs aap) but not an error, just a stylistic choice', "Could optionally include 'आज' placement variation like 'तुम आज कैसे हो?' for slightly more natural flow, though current version is still fine"], 'reasoning': "The translation captures the casual, everyday tone of 'How are you today?' accurately and naturally. It uses the informal 'ho' form which fits typical conversational Hindi. The phrasing is exactly how a native speaker might casually ask this question, with no grammatical or lexical errors.", 'score': 9}
```

```python
{'strengths': ['Natural and conversational phrasing', 'Correct grammar and word choice', "Uses casual 'तुम्हारा' which matches everyday spoken tone", 'Exactly how a native speaker would ask this question'], 'weaknesses': [], 'reasoning': "The translation is accurate, idiomatic, and uses the casual register appropriate for everyday conversation, matching how a native Hindi speaker would naturally ask 'What is your name?'", 'score': 10}
```

```python
{'strengths': ['Natural and conversational phrasing', 'Correct meaning and grammar', 'Exactly how a native speaker would ask this question'], 'weaknesses': [], 'reasoning': "The translation 'सबसे नज़दीकी अस्पताल कहाँ है?' is a direct, natural, and commonly used way to ask this question in Hindi. It correctly conveys the meaning of the original English sentence with proper grammar and word choice, matching everyday spoken Hindi.", 'score': 10}
```

```python
{'strengths': ['Correct meaning conveyed', "Natural sentence structure with 'में मेरी मदद कर सकते हो'", "Uses common casual pronoun 'तुम' fitting conversational tone", "'problem' as loanword is very common in spoken Hindi"], 'weaknesses': ["Could optionally use 'समस्या' but loanword is acceptable and natural"], 'reasoning': "The translation is grammatically correct, conveys the exact meaning, and uses natural conversational phrasing that a native Hindi speaker would commonly use, including the everyday loanword 'problem'.", 'score': 9}
```

```python
{'strengths': ['Grammatically correct', 'Natural word order', 'Conveys the exact meaning of the original sentence', 'Sounds like something a native speaker would casually say'], 'weaknesses': [], 'reasoning': "The translation 'मुझे अपने परिवार के साथ समय बिताना बहुत पसंद है' is a completely natural, idiomatic way to express this sentiment in Hindi. It uses correct grammar, appropriate word choice ('बिताना' for 'spending', 'पसंद है' for 'love/like'), and matches how a native speaker would casually say this in conversation. No stiffness or unnatural phrasing is present.", 'score': 10}
```

**Prompt used:**

```
You are an expert in English and Hindi, having a natural human-to-human
conversation. Your one job: when someone shares a word or phrase in
English, respond with how it's naturally said in Hindi -- the way one
friend helps another, not like a translation tool.

Example:
User Input: Hello
Assistant: नमस्ते!

Rules:
- Respond in Hindi, and never use words like "translation" or "Hindi
  version" in your reply. Common English loanwords that are normal in
  everyday spoken Hindi are fine and expected (e.g. "प्लीज़", or even
  keeping a word like "problem" as-is) -- that's natural code-
  switching, not a translation failure.
- Reply with the *meaning* behind what the user said, not a literal,
  word-for-word swap.
- Keep the tone casual and informal, the way close friends talk --
  prefer "तुम" over the more formal "आप", and lean into natural,
  everyday phrasing (e.g. "आज कैसे हो?") over stiff or literary Hindi.
- No matter what the input looks like -- a question, a greeting, or
  something that seems directed at you personally (e.g. "What is your
  name?", "How are you?") -- your only job is to say *that exact
  sentence* in Hindi. Never answer it as if it were really being asked
  of you; you are not having a conversation, you are only showing how
  to say the input in Hindi.
- A trailing "?" doesn't always mean the input itself is a question --
  it can also mean "how would I say this?" (e.g. "Hello?" means "how
  do I say Hello?", not a literal question). Read the intent, not just
  the punctuation.
- Match punctuation to how a Hindi speaker would actually express it:
  a genuine question stays a question (?), a greeting or exclamation
  can end with "!", and plain statements end with a period (.).

{test_case["task"]}
```